# Rubik's Illusion Colab Bootstrap

This notebook is the first Colab bridge for the project.
It does **not** run diffusion yet.

Its only job is to prove four things inside a Colab runtime:

1. the repo files are available to the runtime
2. the Python bridge imports correctly
3. the exported Rubik's illusion spec loads correctly
4. the solved and scrambled faces render the same way they do locally


## How To Use This Notebook

The safest path in VS Code + Colab is:

- keep this notebook in the repo locally
- connect the notebook to a Colab runtime
- in the setup cell below, set `REPO_URL` to your git remote
- let the runtime clone the repo into `/content`

If the repo is already present in the runtime, the setup cell will reuse it.


In [ ]:
import os
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())
print('In Colab runtime:', 'COLAB_RELEASE_TAG' in os.environ)


In [ ]:
import subprocess
import sys
from pathlib import Path

# Fill this in once you have pushed the repo somewhere reachable from Colab.
REPO_URL = 'https://github.com/netalondon/rubiks-diffusion-illusion.git'
REPO_DIR = Path('/content/rubiks-diffusion-illusion')

if REPO_DIR.exists():
    print(f'Reusing existing repo at {REPO_DIR}')
else:
    if not REPO_URL:
        raise ValueError(
            'Set REPO_URL in this cell before running it, or manually place the repo at /content/rubiks-diffusion-illusion'
        )
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])

requirements_path = REPO_DIR / 'requirements.txt'
if requirements_path.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', REPO_DIR)


In [ ]:
import json
from pathlib import Path

from python_bridge.rubiks_illusion_operator import load_source_faces, render_all_arrangements

spec_path = REPO_DIR / 'public' / 'generated' / 'rubiks-illusion-spec.json'
source_dir = REPO_DIR / 'src' / 'assets' / 'face-art'

spec = json.loads(spec_path.read_text())
source_faces = load_source_faces(source_dir, spec['primeImages'])
rendered = render_all_arrangements(spec, source_faces)

print('Loaded prime images:', spec['primeImages'])
print('Rendered arrangements:', list(rendered.keys()))
print('Solved faces:', list(rendered['solved'].keys()))
print('Scrambled faces:', list(rendered['scrambled'].keys()))


In [ ]:
from IPython.display import display

for arrangement_name in ['solved', 'scrambled']:
    print(arrangement_name.upper())
    for face_id in ['U', 'D', 'L', 'R', 'F', 'B']:
        print(face_id)
        display(rendered[arrangement_name][face_id])


In [ ]:
from python_bridge.rubiks_illusion_operator import save_rendered_faces

output_root = REPO_DIR / 'output' / 'colab-bootstrap-render'
save_rendered_faces(rendered['solved'], output_root / 'solved')
save_rendered_faces(rendered['scrambled'], output_root / 'scrambled')

print('Saved outputs to', output_root)


## Success Condition

If this notebook runs successfully, then the Colab runtime can already do the most important non-ML part:
turn six source images into solved and scrambled derived faces using the same Rubik's operator as the local app.

That is the exact point where we can start replacing fixed source images with optimized diffusion outputs.
